# VIMOP — Entrenamiento del modelo de ruedas (YOLO-pose) en Google Colab

Entrena el modelo de **2 puntos clave** (rueda trasera y rueda delantera) con la GPU gratuita de Colab.
El avance se guarda **en tu Drive**: si la sesión se corta, vuelve a ejecutar todo y **retoma** donde quedó.

**Antes de empezar**
1. Exporta el dataset con `tools/label_wheels.py` (botón *Exportar dataset*): genera `wheels_dataset.zip`.
2. Sube ese `.zip` a tu Drive, en la carpeta `MyDrive/VIMOP/` (o cambia `DRIVE_DIR` abajo).
3. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
4. Ejecuta las celdas en orden (**Entorno de ejecución → Ejecutar todo**).

**Al terminar** descarga `wheels_pose.pt` de tu Drive (`MyDrive/VIMOP/wheels_pose.pt`) y cópialo a `models/wheels_pose.pt`
del proyecto. VIMOP lo detecta solo y muestra *Ruedas: modelo entrenado*.

## 1. Configuración

In [ ]:
# ---- Ajusta solo esto ----------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/VIMOP"      # carpeta de Drive con el .zip; aquí se guarda el avance
DATASET_ZIP = "wheels_dataset.zip"              # nombre del .zip exportado por label_wheels.py
MODEL = "yolo11n-pose"                          # "yolo11n-pose" (rápido) o "yolo11s-pose" (más preciso)
EPOCHS = 100
IMGSZ = 960
BATCH = 16                                      # en una T4 (16 GB) cabe; si da "out of memory", baja a 8
GRAY_PROB = 0.25                                # fracción de imágenes que se pasan a grises (simula infrarrojo)
CAMERA_AUG = 1.0                                # fuerza de los aumentos tipo cámara (ojo de pez, baja resolución, desenfoque, JPEG, ruido, noche); 0 = apagados
RUN_NAME = "wheels"

## 2. Drive y GPU

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Instalar Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 4. Descomprimir el dataset

Se copia al disco local de Colab (mucho más rápido que leer desde Drive en cada época).

In [ ]:
import glob
import shutil
import zipfile

zip_path = os.path.join(DRIVE_DIR, DATASET_ZIP)
assert os.path.exists(zip_path), f"No encuentro {zip_path}. Sube el .zip a esa carpeta de Drive."
LOCAL = "/content/wheels_data"
shutil.rmtree(LOCAL, ignore_errors=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(LOCAL)
DATA_YAML = sorted(glob.glob(f"{LOCAL}/**/dataset.yaml", recursive=True))[0]
print("dataset:", DATA_YAML)
print("imágenes train:", len(glob.glob(os.path.join(os.path.dirname(DATA_YAML), "images/train/*"))))
print("imágenes val:  ", len(glob.glob(os.path.join(os.path.dirname(DATA_YAML), "images/val/*"))))

## 5. Entrenador con aumentos tipo cámara

Las cámaras de seguridad ven motos **pequeñas, borrosas, comprimidas y con las líneas curvas por la lente**, y de noche pasan a **infrarrojo (escala de grises)**. Este entrenador imita eso sobre cada lote: ojo de pez (mueve también cajas y puntos), baja resolución, desenfoque, compresión JPEG, ruido y noche, más la fracción en grises y el brillo variable. *(Es el mismo código de `tools/wheel_training.py` del proyecto.)*

In [ ]:
import math
import random

import numpy as np
import torch
import torch.nn.functional as F


def fisheye_points(xy, k):
    """Puntos (..., 2) en 0–1 de la imagen original -> dónde caen tras el ojo de pez de fuerza k (coincide con fisheye_image)."""
    s = np.asarray(xy, dtype=np.float64) * 2.0 - 1.0
    r_s = np.linalg.norm(s, axis=-1, keepdims=True)
    r = r_s.copy()
    for _ in range(12):                                       # Newton: r (1 + k r²) / (1 + k) = r_s
        g = r * (1 + k * r * r) / (1 + k) - r_s
        r = r - g * (1 + k) / (1 + 3 * k * r * r)
    scale = np.where(r_s > 1e-9, r / np.maximum(r_s, 1e-9), 1.0 + k)
    return (s * scale + 1.0) / 2.0


def fisheye_boxes(xywh, k):
    """Cajas (N, 4) cx, cy, w, h en 0–1 -> la caja que contiene a la caja ya distorsionada."""
    t = np.linspace(0.0, 1.0, 9)
    out = np.empty((len(xywh), 4), dtype=np.float64)
    for i, (cx, cy, bw, bh) in enumerate(np.asarray(xywh, dtype=np.float64)):
        x1, x2, y1, y2 = cx - bw / 2, cx + bw / 2, cy - bh / 2, cy + bh / 2
        xs, ys = x1 + (x2 - x1) * t, y1 + (y2 - y1) * t
        edge = np.concatenate([np.stack([xs, np.full_like(t, y1)], 1), np.stack([xs, np.full_like(t, y2)], 1),
                               np.stack([np.full_like(t, x1), ys], 1), np.stack([np.full_like(t, x2), ys], 1)])
        p = fisheye_points(edge, k)
        lo, hi = p.min(axis=0), p.max(axis=0)
        out[i] = [(lo[0] + hi[0]) / 2, (lo[1] + hi[1]) / 2, hi[0] - lo[0], hi[1] - lo[1]]
    return out


def fisheye_image(img, k):
    """img (3, H, W): efecto ojo de pez (el centro se agranda y los bordes se comprimen). k ~ 0.1 suave, 0.4 fuerte."""
    _, h, w = img.shape
    ys = ((torch.arange(h, device=img.device, dtype=img.dtype) + 0.5) / h) * 2 - 1
    xs = ((torch.arange(w, device=img.device, dtype=img.dtype) + 0.5) / w) * 2 - 1
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    f = (1 + k * (gx * gx + gy * gy)) / (1 + k)
    grid = torch.stack((gx * f, gy * f), dim=-1)[None]
    return F.grid_sample(img[None], grid, mode="bilinear", padding_mode="border", align_corners=False)[0]


def low_resolution(img, factor):
    """Achica la imagen a `factor` (0–1) y la vuelve a agrandar: una moto lejana con pocos píxeles."""
    _, h, w = img.shape
    small = F.interpolate(img[None], scale_factor=factor, mode="area")
    return F.interpolate(small, size=(h, w), mode="bilinear", align_corners=False)[0]


def gaussian_blur(img, sigma):
    r = max(1, int(math.ceil(3 * sigma)))
    x = torch.arange(-r, r + 1, device=img.device, dtype=img.dtype)
    k1 = torch.exp(-(x * x) / (2 * sigma * sigma))
    k1 = k1 / k1.sum()
    c = img.shape[0]
    pad = F.pad(img[None], (r, r, r, r), mode="reflect")
    out = F.conv2d(pad, k1.view(1, 1, 1, -1).repeat(c, 1, 1, 1), groups=c)
    return F.conv2d(out, k1.view(1, 1, -1, 1).repeat(c, 1, 1, 1), groups=c)[0]


def jpeg_roundtrip(img, quality):
    """Comprime y descomprime como JPEG de calidad `quality` (los cuadritos de un video comprimido)."""
    import cv2
    arr = (img.clamp(0, 1) * 255).round().byte().permute(1, 2, 0).cpu().numpy()
    ok, enc = cv2.imencode(".jpg", cv2.cvtColor(arr, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    if not ok:
        return img
    dec = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    return torch.from_numpy(dec).to(img.device).permute(2, 0, 1).to(img.dtype) / 255.0


def _warp_labels(batch, i, k):
    """Mueve las cajas y los puntos de la imagen i del lote con el mismo ojo de pez que se le aplicó a la imagen."""
    sel = batch["batch_idx"] == i
    if not bool(sel.any()):
        return
    boxes = batch["bboxes"]
    boxes[sel] = torch.as_tensor(fisheye_boxes(boxes[sel].double().cpu().numpy(), k), dtype=boxes.dtype,
                                 device=boxes.device)
    kpts = batch.get("keypoints")
    if kpts is not None and kpts.shape[-1] >= 3:
        kp = kpts[sel].double().cpu().numpy()
        moved = fisheye_points(kp[..., :2].reshape(-1, 2), k).reshape(kp[..., :2].shape)
        kp[..., :2] = np.where((kp[..., 2] > 0)[..., None], moved, kp[..., :2])   # los "fuera de imagen" no se mueven
        kpts[sel] = torch.as_tensor(kp, dtype=kpts.dtype, device=kpts.device)


def camera_augment_batch(batch, strength=1.0):
    """
    Aumentos que imitan la cámara fija real, imagen por imagen (cada una se aplica con su probabilidad × strength;
    strength 0 = nada). Orden: ojo de pez (mueve las etiquetas), baja resolución, desenfoque, JPEG, ruido, noche.
    """
    if strength <= 0:
        return
    img = batch["img"]                                            # (B, 3, H, W) float 0–1
    dev = img.device

    def hit(p):
        return random.random() < min(1.0, p * strength)

    for i in range(img.shape[0]):
        x = img[i]
        if hit(0.30):
            k = random.uniform(0.10, 0.40)
            x = fisheye_image(x, k)
            _warp_labels(batch, i, k)
        if hit(0.35):
            x = low_resolution(x, random.uniform(0.25, 0.60))
        if hit(0.30):
            x = gaussian_blur(x, random.uniform(0.6, 2.2))
        if hit(0.30):
            x = jpeg_roundtrip(x, random.randint(15, 60))
        if hit(0.25):
            x = x + torch.randn_like(x) * random.uniform(0.01, 0.05)
        if hit(0.12):                                              # noche / infrarrojo: oscuro, gris y con grano
            x = x.mean(dim=0, keepdim=True).expand(3, -1, -1) * random.uniform(0.25, 0.55)
            x = x + torch.randn(1, *x.shape[1:], device=dev, dtype=x.dtype) * random.uniform(0.03, 0.07)
        img[i] = x.clamp(0.0, 1.0)

from ultralytics.models.yolo.pose import PoseTrainer


class WheelTrainer(PoseTrainer):
    def preprocess_batch(self, batch):
        batch = super().preprocess_batch(batch)
        camera_augment_batch(batch, CAMERA_AUG)
        img = batch["img"]                                   # (B, 3, H, W) float 0–1
        b = img.shape[0]
        gray = torch.rand(b, device=img.device) < GRAY_PROB
        if bool(gray.any()):
            lum = img[gray].mean(dim=1, keepdim=True)
            img[gray] = lum.expand(-1, 3, -1, -1)
        gain = 0.6 + 0.8 * torch.rand(b, 1, 1, 1, device=img.device)
        batch["img"] = (img * gain).clamp_(0.0, 1.0)
        return batch

## 6. Entrenar (con avance en Drive)

`project` apunta a Drive, así que `last.pt` se guarda ahí en cada época. Si la sesión se corta, **vuelve a ejecutar desde la
celda 2**: si encuentra `last.pt` retoma el entrenamiento en lugar de empezar de cero.

In [ ]:
from ultralytics import YOLO

last = os.path.join(DRIVE_DIR, "runs", RUN_NAME, "weights", "last.pt")
if os.path.exists(last):
    print("Retomando desde", last)
    try:
        YOLO(last).train(resume=True, trainer=WheelTrainer)
    except Exception as exc:                          # p. ej. el entrenamiento ya había terminado
        print("No hay nada que retomar:", exc)
else:
    model = YOLO(f"{MODEL}.pt")
    results = model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
        project=os.path.join(DRIVE_DIR, "runs"), name=RUN_NAME, exist_ok=True,
        patience=30, hsv_v=0.5, hsv_s=0.5, degrees=5.0, scale=0.4, fliplr=0.5, mosaic=1.0,
        trainer=WheelTrainer,
    )

## 7. Dejar `best.pt` en Drive

In [ ]:
best = os.path.join(DRIVE_DIR, "runs", RUN_NAME, "weights", "best.pt")
assert os.path.exists(best), "No se encontró best.pt: revisa que el entrenamiento haya terminado."
target = os.path.join(DRIVE_DIR, "wheels_pose.pt")
shutil.copy2(best, target)
print("Listo:", target)
print("Descárgalo y cópialo a  models/wheels_pose.pt  en el proyecto VIMOP.")

## 8. (Opcional) Ver un resultado

Prueba el modelo con una imagen de validación: debe dibujar los dos puntos sobre las ruedas.

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

model = YOLO(target)
val_dir = os.path.join(os.path.dirname(DATA_YAML), "images", "val")
sample = sorted(glob.glob(os.path.join(val_dir, "*")))[:3]
for path in sample:
    res = model.predict(path, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    cv2_imshow(res.plot())